# الفصل 25: تقطير المعرفة: Knowledge Distillation

> نموذج ضخم (المُعلِّم/Teacher) يمتلك معرفة غنية، لكن تشغيله في الإنتاج مكلف وبطيء.

- **الجزء:** 6 — إعادة استخدام النماذج
- **الفصل:** 25
- **المستوى:** متقدم
- **زمن القراءة:** 15 دقيقة
- **المتطلبات:** الفصل 24: التدريب المسبق المستمر: CPT
- **الوسوم:** Distillation، KL Divergence، Temperature، Reasoning Distillation

## كيف تشغّل هذا الدفتر؟

1. **Google Colab:** ارفع الملف عبر `File → Upload notebook`، ثم `Runtime → Run all`.
2. **Kaggle:** أنشئ Notebook ثم `File → Import notebook`، ثم `Run All`.
3. شغّل خلية **إعداد البيئة** أولًا؛ ستفحص الحزم وتثبّت الناقص تلقائيًا.
4. للتدريب أو النماذج الكبيرة: Colab `Runtime → Change runtime type → T4 GPU`، وKaggle `Settings → Accelerator → GPU`.


<div dir="rtl" style="margin:2px 0 18px;line-height:2.4;">
<b>افتح هذا الدفتر على منصة مجانية:</b><br>
<a href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-06-model-reuse/ch25-knowledge-distillation.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#F9AB00;color:#111111;">Google Colab</a> <a href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-06-model-reuse%2Fch25-knowledge-distillation.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#20BEFF;color:#FFFFFF;">Kaggle</a> <a href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-06-model-reuse/ch25-knowledge-distillation.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#579ACA;color:#FFFFFF;">Binder</a> <a href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-06-model-reuse/ch25-knowledge-distillation.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#6C5CE7;color:#FFFFFF;">Gradient</a> <a href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-06-model-reuse/ch25-knowledge-distillation.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#37474F;color:#FFFFFF;">nbviewer</a> <a href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-06-model-reuse/ch25-knowledge-distillation.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#24292F;color:#FFFFFF;">GitHub</a>
</div>


## ملاحظات التشغيل

- **الحزم:** torch, transformers
- **GPU:** يستخدم هذا الدفتر PyTorch أو نماذج كبيرة؛ يُفضّل GPU. يمكنك البدء بنماذج صغيرة على CPU، لكن التدريب الكامل سيكون بطيئًا.
- **بيانات/نماذج خارجية:** بعض الخلايا تتوقع ملفات أو نماذج محمّلة مسبقًا. عدّل المسارات أو حمّل البيانات قبل التشغيل.
- **خلايا ثقيلة:** قد تستغرق خلايا التدريب/التحميل وقتًا طويلًا. شغّلها خلية بخية، وقلّل `batch_size` أو استخدم نموذجًا أصغر عند الحاجة.
- **مقاطع توضيحية:** بعض الشيفرة تحتوي `...` أو `NotImplementedError`؛ هذا مقصود لبيان البنية وليس خطأً في الدفتر.


<style>
/* نص عربي من اليمين إلى اليسار، مع إبقاء الشيفرة من اليسار */
.rendered_html, .jp-RenderedHTMLCommon, .text_cell_render, .markdown-body {
  direction: rtl;
  text-align: right;
}
.rendered_html pre, .rendered_html code,
.jp-RenderedHTMLCommon pre, .jp-RenderedHTMLCommon code,
.text_cell_render pre, .text_cell_render code,
.markdown-body pre, .markdown-body code {
  direction: ltr;
  text-align: left;
  unicode-bidi: embed;
}
.rendered_html table, .jp-RenderedHTMLCommon table, .text_cell_render table {
  direction: rtl;
}
</style>


In [ ]:
# === إعداد البيئة ===
# يفحص هذا القسم الحزم المطلوبة ويحاول تثبيت الناقص منها تلقائيًا.
# شغّل هذه الخلية أولًا، ثم بقية خلايا الدفتر.
import importlib.util
import subprocess
import sys

print("Python:", sys.version.split()[0])

# (اسم وحدة الاستيراد، اسم حزمة pip)
REQUIRED = [
    ("torch", "torch"),
    ("transformers", "transformers")
]
AUTO_INSTALL = {"PIL", "bs4", "datasets", "faiss", "fasttext", "librosa", "networkx", "opentelemetry", "peft", "prometheus_client", "pydantic", "rank_bm25", "requests", "sentence_transformers", "sentencepiece", "soundfile", "tokenizers", "trafilatura", "transformers", "wandb", "yaml"}


def has_module(name):
    try:
        return importlib.util.find_spec(name) is not None
    except Exception:
        return False


if not REQUIRED:
    print("لا توجد مكتبات خارجية مطلوبة؛ يكفي Python القياسي.")
else:
    missing = [(module, package) for module, package in REQUIRED if not has_module(module)]
    auto_specs = []
    manual_specs = []
    for module, package in missing:
        specs = package.split()
        if module in AUTO_INSTALL:
            auto_specs.extend(specs)
        else:
            manual_specs.extend(specs)
    auto_specs = list(dict.fromkeys(auto_specs))
    manual_specs = list(dict.fromkeys(manual_specs))

    if auto_specs:
        print("⏳ تثبيت تلقائي:", " ".join(auto_specs))
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *auto_specs])
            print("✅ تم التثبيت.")
        except Exception as exc:
            print("⚠️ تعذّر التثبيت التلقائي:", exc)
            print("شغّل يدويًا: %pip install", " ".join(auto_specs))
    if manual_specs:
        print("⚠️ حزم تحتاج تثبيتًا يدويًا:", " ".join(manual_specs))
        print("شغّل: %pip install", " ".join(manual_specs))
    if not auto_specs and not manual_specs:
        print("✅ كل الحزم المطلوبة متاحة.")

    print("\nالحالة النهائية:")
    for module, package in REQUIRED:
        print(("✅" if has_module(module) else "⚠️"), package)


<div class="notebook-actions" dir="rtl"><span class="notebook-actions__label">شغّل الدفتر:</span><a class="nb-action nb-action--colab" href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-06-model-reuse/ch25-knowledge-distillation.ipynb" target="_blank" rel="noopener" title="افتح في Google Colab — GPU مجاني">Google Colab</a><a class="nb-action nb-action--kaggle" href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-06-model-reuse%2Fch25-knowledge-distillation.ipynb" target="_blank" rel="noopener" title="افتح في Kaggle — GPU/TPU مجاني">Kaggle</a><a class="nb-action nb-action--binder" href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-06-model-reuse/ch25-knowledge-distillation.ipynb" target="_blank" rel="noopener" title="افتح في Binder — بيئة Jupyter مجانية">Binder</a><a class="nb-action nb-action--gradient" href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-06-model-reuse/ch25-knowledge-distillation.ipynb" target="_blank" rel="noopener" title="افتح في Paperspace Gradient — GPU مجاني محدود">Gradient</a><a class="nb-action nb-action--nbviewer" href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-06-model-reuse/ch25-knowledge-distillation.ipynb" target="_blank" rel="noopener" title="اعرض الدفتر دون تنفيذ">nbviewer</a><a class="nb-action nb-action--github" href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-06-model-reuse/ch25-knowledge-distillation.ipynb" target="_blank" rel="noopener" title="اعرض الملف المصدر على GitHub">GitHub</a></div>

> **ملخص الفصل**
>
> نموذج ضخم (المُعلِّم/Teacher) يمتلك معرفة غنية، لكن تشغيله في الإنتاج مكلف وبطيء.

> **بيانات الفصل**
>
> - **الجزء:** 6 — إعادة استخدام النماذج
> - **المستوى:** متقدم
> - **زمن القراءة:** 15 دقيقة
> - **المتطلبات:** الفصل 24: التدريب المسبق المستمر: CPT


## مخرجات التعلم

بنهاية هذا الفصل ستكون قادرًا على:

- فهم التقطير القائم على الاستجابة: الأساس الرياضي الكامل
- فهم التقطير القائم على الميزات (Feature-Based Distillation)
- فهم تقطير سلاسل التفكير (Reasoning/Chain-of-Thought Distillation)


## 1. لماذا يهمّك هذا الموضوع؟

🟢 طبقة الحدس
نموذج ضخم (المُعلِّم/Teacher) يمتلك معرفة غنية، لكن تشغيله في الإنتاج مكلف وبطيء. ماذا لو استطعنا "عصر" هذه المعرفة في نموذج أصغر بكثير (التلميذ/Student) يحتفظ بمعظم القدرة لكن بكسر من التكلفة؟ هذا بالضبط ما يفعله Knowledge Distillation: بدل تعليم التلميذ من "الإجابات الصحيحة الخام" فقط (كما في SFT التقليدي)، نُعلِّمه من التوزيع الاحتمالي الكامل الذي ينتجه المُعلِّم — وهذا التوزيع يحمل معلومات أغنى بكثير ("المعرفة المظلمة" / Dark Knowledge) من مجرد التسمية الصحيحة الواحدة.


## 2. الخريطة الذهنية


```text
أنواع التقطير المعرفي الثلاثة

1. Response-Based Distillation (الأكثر شيوعًا)
   التلميذ يُقلِّد توزيع الاحتمالات النهائي للمُعلِّم (logits/soft labels)

2. Feature-Based Distillation
   التلميذ يُقلِّد التمثيلات الداخلية (hidden states) للمُعلِّم
   في طبقات وسيطة محددة

3. Relation-Based Distillation
   التلميذ يُقلِّد العلاقات بين عدة عيّنات (مثل: مصفوفة التشابه
   بين تمثيلات دفعة من العيّنات) وليس فقط تمثيل كل عيّنة منفردة
```


## 3. الشرح المتدرج

### 3.1 التقطير القائم على الاستجابة: الأساس الرياضي الكامل

🟡 طبقة الآلية

الفكرة الأصلية من ورقة Hinton et al. (2015) Distilling the Knowledge in a Neural Network:


```text
بدل التدريب العادي (Hard Labels فقط):
L_hard = CrossEntropy(student_logits, true_label)

نُضيف خسارة تقطير (Soft Labels من المُعلِّم):
L_soft = KL_Divergence(
    softmax(student_logits / T),
    softmax(teacher_logits / T)
) × T²

L_total = α × L_hard + (1 - α) × L_soft
لماذا درجة الحرارة T (Temperature)؟

🔴 طبقة التعمّق
```


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class KnowledgeDistillationLoss(nn.Module):
    """
    التطبيق الكامل لخسارة التقطير المعرفي الكلاسيكية (Hinton et al., 2015)
    مع دمج Hard Label Loss وSoft Label Loss، قابلة للتطبيق مباشرة على
    نماذج اللغة الكبيرة (Response-Based Distillation على مستوى الرمز).
    """
    def __init__(self, temperature: float = 4.0, alpha: float = 0.3):
        super().__init__()
        self.temperature = temperature
        self.alpha = alpha  # وزن Hard Loss؛ (1-alpha) لـ Soft Loss

    def forward(self, student_logits: torch.Tensor,
                teacher_logits: torch.Tensor,
                true_labels: torch.Tensor,
                attention_mask: torch.Tensor = None) -> dict:
        """
        student_logits, teacher_logits: [batch, seq_len, vocab_size]
        true_labels: [batch, seq_len] (رموز الهدف الحقيقية - Hard Labels)
        """
        vocab_size = student_logits.size(-1)

        # --- 1. Hard Label Loss (كما في SFT عادي) ---
        hard_loss = F.cross_entropy(
            student_logits.view(-1, vocab_size),
            true_labels.view(-1),
            ignore_index=-100,  # لتجاهل رموز الحشو (padding)
            reduction="mean",
        )

        # --- 2. Soft Label Loss (جوهر التقطير - KL Divergence) ---
        student_log_probs = F.log_softmax(
            student_logits / self.temperature, dim=-1
        )
        teacher_probs = F.softmax(
            teacher_logits / self.temperature, dim=-1
        )
        # يجب فصل تدرج المُعلِّم - لا نُحدِّث أوزانه إطلاقًا
        teacher_probs = teacher_probs.detach()

        # KL Divergence لكل رمز، مع تجاهل رموز الحشو عبر attention_mask
        soft_loss_per_token = F.kl_div(
            student_log_probs, teacher_probs, reduction="none"
        ).sum(dim=-1)  # [batch, seq_len]

        if attention_mask is not None:
            soft_loss = (soft_loss_per_token * attention_mask).sum() / attention_mask.sum()
        else:
            soft_loss = soft_loss_per_token.mean()

        # الضرب بـ T² ضروري لموازنة مقياس التدرجات (كما أثبت Hinton et al.)
        # لأن تدرجات soft loss تتناسب عكسيًا مع T²
        soft_loss = soft_loss * (self.temperature ** 2)

        # --- 3. الدمج ---
        total_loss = self.alpha * hard_loss + (1 - self.alpha) * soft_loss

        return {
            "total_loss": total_loss,
            "hard_loss": hard_loss.item(),
            "soft_loss": soft_loss.item(),
        }

class ResponseBasedDistillationTrainer:
    """خط أنابيب تدريب كامل للتقطير القائم على الاستجابة لنماذج اللغة."""
    def __init__(self, teacher_model, student_model, tokenizer,
                 temperature: float = 4.0, alpha: float = 0.3):
        self.teacher = teacher_model.eval()  # المُعلِّم دائمًا في وضع التقييم
        self.student = student_model
        self.tokenizer = tokenizer
        self.loss_fn = KnowledgeDistillationLoss(temperature, alpha)
        self.optimizer = torch.optim.AdamW(self.student.parameters(), lr=2e-5)

        # تجميد كامل لأوزان المُعلِّم - توفير ذاكرة وضمان عدم التحديث
        for param in self.teacher.parameters():
            param.requires_grad = False

    def train_step(self, input_ids, attention_mask, labels):
        with torch.no_grad():
            teacher_outputs = self.teacher(
                input_ids=input_ids, attention_mask=attention_mask
            )
            teacher_logits = teacher_outputs.logits

        student_outputs = self.student(
            input_ids=input_ids, attention_mask=attention_mask
        )
        student_logits = student_outputs.logits

        losses = self.loss_fn(
            student_logits, teacher_logits, labels, attention_mask
        )

        self.optimizer.zero_grad()
        losses["total_loss"].backward()
        torch.nn.utils.clip_grad_norm_(self.student.parameters(), 1.0)
        self.optimizer.step()

        return losses


> ⚠️ خطأ شائع: استخدام نفس Tokenizer للمُعلِّم والتلميذ أمر إلزامي في Response-Based Distillation على مستوى الرمز — إذا اختلف المفردات (vocabulary) بين النموذجين، توزيعات الاحتمالات ستكون على فضاءات مختلفة تمامًا ولا يمكن مقارنتها مباشرة بـ KL Divergence. إذا أردت تقطير معرفة بين نموذجين بمفردات مختلفة، يجب اللجوء لـ Feature-Based Distillation أو محاذاة المفردات يدويًا (أكثر تعقيدًا بكثير).

### 3.2 التقطير القائم على الميزات (Feature-Based Distillation)

🟡 طبقة الآلية

بدل الاكتفاء بمطابقة المخرجات النهائية فقط، نُجبر التمثيلات الداخلية (Hidden States) للتلميذ في طبقات وسيطة محددة على التشابه مع تمثيلات المُعلِّم المقابلة — هذا النهج استخدمته DistilBERT وTinyBERT بنجاح كبير.


In [ ]:
class FeatureBasedDistillationLoss(nn.Module):
    """
    تقطير قائم على الميزات (بأسلوب TinyBERT): مطابقة Hidden States
    بين طبقات مُحدَّدة في المُعلِّم والتلميذ، مع إسقاط خطي (Linear
    Projection) للتعويض عن اختلاف الأبعاد بين النموذجين.
    """
    def __init__(self, teacher_hidden_dim: int, student_hidden_dim: int,
                 layer_mapping: dict):
        """
        layer_mapping: قاموس يُحدِّد أي طبقة من التلميذ تُقارَن بأي طبقة
        من المُعلِّم. مثال: {0: 0, 1: 4, 2: 8, 3: 12} يعني طبقة التلميذ 0
        تُقارَن بطبقة المُعلِّم 0، وطبقة التلميذ 1 تُقارَن بطبقة المُعلِّم 4
        (تخطي طبقات - شائع عند ضغط نموذج 12 طبقة إلى 4 طبقات مثلًا)
        """
        super().__init__()
        self.layer_mapping = layer_mapping
        # إسقاط خطي لمحاذاة أبعاد التلميذ (أصغر عادة) مع أبعاد المُعلِّم
        self.projections = nn.ModuleDict({
            str(s_layer): nn.Linear(student_hidden_dim, teacher_hidden_dim)
            for s_layer in layer_mapping.keys()
        })

    def forward(self, student_hidden_states: list, teacher_hidden_states: list):
        """
        student_hidden_states, teacher_hidden_states: قوائم من تنسورات
        [batch, seq_len, hidden_dim] لكل طبقة من طبقات كل نموذج.
        """
        total_feature_loss = 0.0
        for student_layer_idx, teacher_layer_idx in self.layer_mapping.items():
            student_hidden = student_hidden_states[student_layer_idx]
            teacher_hidden = teacher_hidden_states[teacher_layer_idx].detach()

            # إسقاط تمثيل التلميذ لنفس بُعد المُعلِّم
            projected_student = self.projections[str(student_layer_idx)](
                student_hidden
            )

            # خسارة MSE بين التمثيلين المُحاذَيين
            layer_loss = F.mse_loss(projected_student, teacher_hidden)
            total_feature_loss += layer_loss

        return total_feature_loss / len(self.layer_mapping)

class AttentionBasedDistillationLoss(nn.Module):
    """
    تقطير خرائط الانتباه نفسها (Attention Map Distillation - أسلوب
    TinyBERT المتقدم): إجبار التلميذ على تعلّم "أين ينظر" المُعلِّم،
    وليس فقط ماذا ينتج - هذا ينقل معرفة بنيوية أعمق عن العلاقات
    اللغوية بين الرموز.
    """
    def __init__(self, layer_mapping: dict):
        super().__init__()
        self.layer_mapping = layer_mapping

    def forward(self, student_attentions: list, teacher_attentions: list):
        """
        student_attentions, teacher_attentions: قوائم تنسورات
        [batch, n_heads, seq_len, seq_len] لكل طبقة.
        """
        total_attn_loss = 0.0
        for student_layer_idx, teacher_layer_idx in self.layer_mapping.items():
            student_attn = student_attentions[student_layer_idx]
            teacher_attn = teacher_attentions[teacher_layer_idx].detach()

            # قد يختلف عدد الرؤوس بين النموذجين - متوسط عبر الرؤوس كحل مبسّط
            student_attn_avg = student_attn.mean(dim=1)
            teacher_attn_avg = teacher_attn.mean(dim=1)

            # MSE بين خرائط الانتباه (بعد قص القيم الصغيرة جدًا لتفادي عدم استقرار log)
            attn_loss = F.mse_loss(student_attn_avg, teacher_attn_avg)
            total_attn_loss += attn_loss

        return total_attn_loss / len(self.layer_mapping)


### 3.3 تقطير سلاسل التفكير (Reasoning/Chain-of-Thought Distillation)

🔴 طبقة التعمّق

مع ظهور نماذج الاستدلال (الفصل 60)، برز نوع جديد مهم جدًا من التقطير: تقطير القدرة على الاستدلال نفسها، وليس فقط الإجابة النهائية. DeepSeek-R1 استخدمت هذا فعليًا لتوليد سلسلة من نماذج أصغر (Qwen وLlama بأحجام 1.5B-70B) تحمل قدرات استدلال قوية جدًا رغم صغر حجمها، عبر Rejection Sampling من R1 الكبير.


In [ ]:
class ReasoningDistillationPipeline:
    """
    خط أنابيب لتقطير قدرة الاستدلال من نموذج استدلال كبير (مثل
    DeepSeek-R1) إلى نموذج أصغر، بأسلوب SFT على سلاسل تفكير مُولَّدة
    من المُعلِّم (وليس عبر RL مباشرة على التلميذ - أرخص بكثير).
    """
    def __init__(self, teacher_reasoning_model, student_model,
                 verification_fn):
        self.teacher = teacher_reasoning_model
        self.student = student_model
        self.verify = verification_fn  # دالة تحقق قابلة للبرمجة (الفصل 60)

    def generate_verified_reasoning_dataset(self, problems: list,
                                             n_samples_per_problem: int = 4) -> list:
        """
        توليد سلاسل تفكير من المُعلِّم لكل مسألة، والاحتفاظ فقط بالسلاسل
        التي تُفضي لإجابة صحيحة (Rejection Sampling) - هذا يضمن أن
        التلميذ يتعلّم فقط من أنماط تفكير "ناجحة" فعليًا، لا عشوائية.
        """
        verified_dataset = []
        for problem in problems:
            correct_reasoning_found = False
            for _ in range(n_samples_per_problem):
                reasoning_trace = self.teacher.generate(
                    problem["question"], temperature=0.7, max_tokens=4096
                )
                is_correct = self.verify(reasoning_trace, problem["ground_truth"])
                if is_correct:
                    verified_dataset.append({
                        "question": problem["question"],
                        "reasoning_trace": reasoning_trace,
                    })
                    correct_reasoning_found = True
                    break  # نكتفي بأول سلسلة صحيحة لكل مسألة (اقتصاد في الحوسبة)

            if not correct_reasoning_found:
                # تجاهل المسائل التي فشل المُعلِّم نفسه في حلها بشكل صحيح
                continue

        return verified_dataset

    def distill_via_sft(self, verified_dataset: list, num_epochs: int = 3):
        """
        التقطير هنا يتم عبر SFT عادي (الجزء 7) على سلاسل التفكير
        المُولَّدة والمُتحقَّق منها - وليس عبر مطابقة logits كما في
        القسم 3.1. هذا النهج أثبت فعالية مذهلة في تقارير DeepSeek-R1:
        نموذج Qwen 7B مُقطَّر بهذه الطريقة تفوّق على بعض النماذج
        الأكبر بكثير المُدرَّبة بـ RL مباشرة عليها.
        """
        for epoch in range(num_epochs):
            for example in verified_dataset:
                full_text = (
                    f"السؤال: {example['question']}\n"
                    f"{example['reasoning_trace']}"
                )
                # تدريب SFT قياسي - راجع الجزء السابع للتفاصيل الكاملة
                self._sft_step(full_text)

    def _sft_step(self, text: str):
        raise NotImplementedError("يُطبَّق عبر حلقة SFT قياسية - راجع الجزء السابع")


🎯 نتيجة لافتة من تقرير DeepSeek-R1: نموذج DeepSeek-R1-Distill-Qwen-7B (مُقطَّر بهذه المنهجية SFT البسيطة من R1) تفوّق في بعض معايير الاستدلال الرياضي على QwQ-32B-Preview (نموذج أكبر بأربع مرات ومُدرَّب بـ RL مباشرة عليه)! هذا يُظهر أن تقطير سلاسل تفكير عالية الجودة قد يكون أحيانًا أكثر كفاءة من إعادة تدريب RL كامل من الصفر على نموذج صغير — RL على نماذج صغيرة قد لا "يكتشف" أنماط الاستدلال المتقدمة بنفس سهولة نموذج كبير يمتلك قاعدة معرفية أوسع.


## 4. مثال محلول خطوة بخطوة


In [ ]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
teacher = GPT2LMHeadModel.from_pretrained("gpt2-large")   # 774M معامل
student = GPT2LMHeadModel.from_pretrained("gpt2")         # 124M معامل

trainer = ResponseBasedDistillationTrainer(
    teacher_model=teacher, student_model=student, tokenizer=tokenizer,
    temperature=4.0, alpha=0.3,
)

# محاكاة خطوة تدريب واحدة


text = "الذكاء الاصطناعي هو مجال يهتم بـ"
inputs = tokenizer(text, return_tensors="pt")
labels = inputs["input_ids"].clone()

losses = trainer.train_step(
    input_ids=inputs["input_ids"],
    attention_mask=inputs["attention_mask"],
    labels=labels,
)
print(f"الخسارة الكلية: {losses['total_loss'].item():.4f}")
print(f"  - Hard Loss (الدقة المباشرة): {losses['hard_loss']:.4f}")
print(f"  - Soft Loss (المعرفة المظلمة): {losses['soft_loss']:.4f}")


## 5. نصائح وحيل

ابدأ بـ T=3-5 وα=0.3-0.5 كقيم افتراضية معقولة، ثم اضبطهما تجريبيًا حسب الفجوة الحالية بين دقة النموذجين.
إذا كان التلميذ أصغر بكثير جدًا من المُعلِّم (فجوة حجم >10×)، فكّر في تقطير تدريجي عبر نموذج وسيط (Teacher Assistant) بدل القفز مباشرة — ورقة Improved Knowledge Distillation via Teacher Assistant تُظهر أن هذا يُحسِّن النتيجة النهائية.
لتقطير سلاسل التفكير، استخدم دائمًا Rejection Sampling (تصفية السلاسل الخاطئة) — تقطير سلاسل تفكير "تبدو منطقية لكنها تُفضي لإجابة خاطئة" يضرّ التلميذ أكثر مما ينفعه.


## 6. أخطاء شائعة ومصائد

تجاهل فصل تدرج المُعلِّم (.detach() أو torch.no_grad()) — خطأ قد يؤدي لتحديث أوزان المُعلِّم بالخطأ أو استهلاك ذاكرة هائلة غير ضرورية لتخزين تدرجاته.
استخدام α=1.0 (الاعتماد الكلي على Hard Labels) مما يُلغي فائدة التقطير بالكامل ويجعله SFT عاديًا لا أكثر.
تجاهل أن المُعلِّم يجب أن يكون في وضع eval() دائمًا (لتعطيل Dropout وBatchNorm العشوائي) أثناء توليد soft labels، وإلا ستكون التسميات اللينة غير مستقرة بين دفعة وأخرى.


## 7. جدول مقارنة

| النوع | ماذا يُطابَق | التعقيد | أفضل استخدام |
|---|---|---|---|
| Response-Based | توزيع الاحتمالات النهائي (logits) | بسيط | عندما يتشارك النموذجان نفس المفردات |
| Feature-Based | التمثيلات الداخلية (hidden states) | متوسط | ضغط عميق (تقليل عدد الطبقات) |
| Attention-Based | خرائط الانتباه | متوسط-عالٍ | نقل فهم بنيوي/نحوي عميق |
| Reasoning/CoT Distillation | سلاسل التفكير الكاملة (عبر SFT) | بسيط تنفيذيًا، مكلف توليديًا | نقل قدرات استدلال لنماذج صغيرة |


## 8. ملخص أفضل الممارسات

✅ استخدم نفس Tokenizer للمُعلِّم والتلميذ عند التقطير القائم على الاستجابة.
✅ جرّب قيم T متعددة (2، 4، 8) وقارن أداء التلميذ النهائي — لا توجد قيمة "صحيحة" عالميًا.
✅ لضغط عميق جدًا (تقليل الطبقات بشكل كبير)، ادمج Feature-Based وResponse-Based معًا للحصول على أفضل نتيجة.
✅ لتقطير الاستدلال، استخدم Rejection Sampling دائمًا ولا تُدرِّب على سلاسل غير مُتحقَّق منها.


## 9. أسئلة مقابلات

**سؤال: لماذا تُضرَب خسارة Soft Label بمربع درجة الحرارة (T²) في معادلة Hinton الأصلية؟**

**إجابة نموذجية:** عند حساب تدرج دالة softmax بالنسبة للـ logits عند قسمتها على T، فإن مقياس (magnitude) التدرج الناتج يتناسب عكسيًا مع T² تقريبًا (يمكن اشتقاق هذا رياضيًا من مشتقة softmax(z/T)). بدون هذا التعويض، فإن رفع T (لتلطيف التوزيع والكشف عن المعرفة المظلمة) سيُقلِّص تلقائيًا مساهمة Soft Loss في التدرج الكلي بشكل كبير جدًا، مما يجعل Hard Loss يهيمن على التدريب بغض النظر عن قيمة α المُختارة. ضرب Soft Loss بـ T² يُعيد موازنة مقياس تدرجاتها لتبقى قابلة للمقارنة مع Hard Loss بغض النظر عن قيمة T المُستخدمة، مما يسمح بضبط T وα بشكل مستقل عن بعضهما.


## 10. تمرين عملي

نفّذ FeatureBasedDistillationLoss على نموذج BERT (12 طبقة) كمُعلِّم ونموذج 4 طبقات كتلميذ، باستخدام layer_mapping = {0: 2, 1: 5, 2: 8, 3: 11}. درّب التلميذ بدمج Response-Based وFeature-Based معًا، وقارن النتيجة النهائية (على مهمة تصنيف نصوص بسيطة) مع تلميذ مُدرَّب بـ Response-Based فقط.


## 11. مراجع للتعمّق

- Hinton et al. (2015): Distilling the Knowledge in a Neural Network
- Sanh et al. (2019): DistilBERT, a distilled version of BERT
- Jiao et al. (2019): TinyBERT: Distilling BERT for Natural Language Understanding
- Mirzadeh et al. (2019): Improved Knowledge Distillation via Teacher Assistant
- DeepSeek-AI (2025): DeepSeek-R1 Technical Report (قسم Distillation)


## 🧪 مساحة تجربة

استخدم الخلية التالية لتجربة أفكار الفصل أو حلّ التمرين. عدّل القيم وأعد تشغيل الخلية.


In [ ]:
# 🧪 مساحة تجربة
# اكتب هنا حلّ التمرين أو أي تجربة إضافية.


---
📘 **المصدر:** كتاب *إتقان المحوّلات والنماذج اللغوية الكبيرة* — إعادة استخدام النماذج / الفصل 25: تقطير المعرفة: Knowledge Distillation.

هذا الدفتر مولّد آليًا من ملف الفصل؛ لا تعدّله يدويًا، بل عدّل المصدر ثم أعد البناء.
